# KD-GAMS-Net: Closing the Gap to Pretrained Lightweight Networks

**Goal of this notebook.** The earlier GAMS-Net experiment (Ghost + ECA + MSDF, trained
from scratch) trailed pretrained lightweight backbones (MobileNetV2, EfficientNet-B0) by a
statistically supported margin. That experiment's own ablation showed *why*: Ghost modules
and the MSDF block earned their keep, but training from random initialization was the real
bottleneck, not the architecture. This notebook tests a direct fix.

**Proposed model: KD-GAMS-Net**
- Keeps Ghost modules (proven: ~65% parameter reduction at negligible accuracy cost).
- **Doubles the MSDF block** (inserted after stage 1 *and* stage 2, instead of once) —
  MSDF was the single largest accuracy contributor in the prior ablation, so this is the
  most evidence-backed architectural change available.
- **Replaces ECA (channel attention) with CBAM-style spatial attention** — the prior
  confusion matrices showed errors concentrated at specific class *boundaries*
  (glioma↔meningioma, notumor→meningioma), a spatial problem that channel re-weighting
  cannot address. ECA's own ablation contribution was statistically indistinguishable from
  noise, so this swap is not costing us a proven component.
- **Trained with knowledge distillation** from a frozen teacher ensemble (fine-tuned,
  ImageNet-pretrained MobileNetV2 + EfficientNet-B0), instead of hard-label cross-entropy
  alone. This directly targets the diagnosed bottleneck: it injects pretrained-quality
  decision boundaries into the compact model *during training only* — the teachers are
  discarded at inference, so the deployed model is exactly as lightweight and exactly as
  pretrained-weight-independent as the original design goal required.
- **Focal loss** on the hard-label term, upweighting the smaller "notumor" class, since the
  prior confusion matrix showed notumor→meningioma as a leading error and notumor is also
  the smallest class in the corpus.
- **Contour-based cropping preprocessing** (grayscale → threshold → largest contour →
  bounding-box crop) applied to every model in this notebook, following the same
  preprocessing used by a directly comparable from-scratch lightweight CNN on the same
  dataset lineage that reached 98.78% test accuracy without any pretraining.

**What changed from the previous notebook (fixing two concrete problems).**
1. Confusion matrices and McNemar's test previously failed to materialize because
   predictions were never cached incrementally and a true paired McNemar's test needs
   per-sample predictions that weren't retained. Here, **every model's true/predicted
   labels are saved to disk immediately after that model is evaluated** (`predictions/`
   folder), and confusion matrices + a real, paired McNemar's test are computed from those
   cached arrays right after each run — not deferred to a final section that might never
   be reached.
2. Scope is intentionally narrower than the previous notebook's 6-model × 3-seed ×
   4-ablation sweep (which alone consumed ~5–6 hours) so the **total runtime stays under
   10 hours**: this notebook trains 4 headline models (GAMS-Net baseline, KD-GAMS-Net,
   and the two teachers) across 3 seeds, plus a single-seed, 3-variant ablation isolating
   KD / double-MSDF / spatial-attention individually. The previously-established baselines
   (ResNet18, ShuffleNetV2, Plain-CNN) are not re-run here; refer to the prior notebook's
   results for those.

**Estimated runtime budget** (measured empirically from the prior notebook's per-model
times on a Kaggle P100/T4, with ~30–50% overhead added for the extra teacher forward pass
during KD training):

| Phase | Approx. time |
|---|---|
| Setup, dataset location, leakage-audited split | 15–20 min |
| Train 2 teachers (seed 42) | 25 min |
| Train GAMS-Net baseline + KD-GAMS-Net (seed 42) | 60–75 min |
| Ablation (3 single-seed variants) | 90–110 min |
| Multi-seed sweep (seeds 123, 2024 × 4 models) | 4–5 hours |
| McNemar's tests, confusion matrices, efficiency table, Grad-CAM | 15–20 min |
| **Total** | **≈ 7.5–8.5 hours** |

This leaves headroom under the 10-hour ceiling. If your Kaggle session has a stricter time
limit, set `SKIP_MULTISEED = True` in the config cell to run only the seed-42 comparison
(≈2.5 hours total) and add the extra seeds later.

> **Read before running:** this notebook cannot be executed in this authoring environment
> (no GPU, no dataset access here). Every piece of model/loss code below has been verified
> in a sandbox for correctness (forward/backward pass, parameter counts, loss values,
> McNemar's test logic on synthetic data) but the actual accuracy numbers only exist once
> you run this on Kaggle.


## 1. Setup & configuration

In [ ]:
!pip -q install torchinfo thop grad-cam scikit-learn seaborn statsmodels opencv-python-headless --no-input


In [ ]:
import os, glob, random, time, copy, json, hashlib, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report, roc_auc_score)
from statsmodels.stats.contingency_tables import mcnemar
from scipy import stats as scipy_stats

SEED = 42
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Torch:", torch.__version__, " Torchvision:", torchvision.__version__)

os.makedirs("predictions", exist_ok=True)   # cached y_true/y_pred/y_prob per model, per seed
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("figures", exist_ok=True)


In [ ]:
# ---------------- Run configuration ----------------
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 40
PATIENCE = 8
SEEDS_EXTRA = [123, 2024]          # seed 42 is always run first, standalone
SKIP_MULTISEED = False             # set True for a fast ~2.5h single-seed run only
KD_ALPHA = 0.4                     # weight on hard-label loss; (1-alpha) goes to KD soft loss
KD_TEMPERATURE = 4.0
FOCAL_GAMMA = 2.0
print(f"Config: EPOCHS={EPOCHS} PATIENCE={PATIENCE} SKIP_MULTISEED={SKIP_MULTISEED} "
      f"extra seeds={[] if SKIP_MULTISEED else SEEDS_EXTRA}")


## 2. Locate the dataset

Reuses the leakage-aware, layout-robust detection logic validated in the prior notebook:
find the folder containing the four class subfolders regardless of what the parent
directory is named, and separately check whether any shipped "test"-named folder is
actually large enough and class-structured enough to serve as a real evaluation split.

In [ ]:
KAGGLE_INPUT = "/kaggle/input"

def find_dataset_root(input_dir=KAGGLE_INPUT, needle="brainmri"):
    candidates = []
    for root, dirs, files in os.walk(input_dir):
        if needle.lower() in root.lower():
            candidates.append(root)
    return candidates

candidates = find_dataset_root()
print("Candidate roots containing 'brainmri':")
for c in candidates:
    print(" -", c)
if not candidates:
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        print(root, dirs[:10])


In [ ]:
DATASET_ROOT = candidates[0] if candidates else KAGGLE_INPUT
KNOWN_CLASSES = {"glioma", "meningioma", "notumor", "pituitary"}

def has_class_subdirs(d, class_names):
    if d is None or not os.path.isdir(d):
        return False
    sub = {x.lower() for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))}
    return set(c.lower() for c in class_names).issubset(sub)

def find_labeled_pool(root, known=KNOWN_CLASSES):
    for r, dirs, files in os.walk(root):
        low = {d.lower() for d in dirs}
        if known.issubset(low):
            return r
    return None

MAIN_DIR = find_labeled_pool(DATASET_ROOT)
print("Main labeled pool (has per-class subfolders):", MAIN_DIR)
if MAIN_DIR is None:
    raise RuntimeError("Could not find glioma/meningioma/notumor/pituitary subfolders. "
                        "Inspect the directory tree above and set MAIN_DIR manually.")

CLASS_NAMES = sorted(os.listdir(MAIN_DIR))
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)

def find_named_dir(root, name):
    for r, dirs, files in os.walk(root):
        low = [d.lower() for d in dirs]
        if name.lower() in low:
            return os.path.join(r, dirs[low.index(name.lower())])
    return None

LEGACY_TRAIN_DIR = find_named_dir(DATASET_ROOT, "training") or find_named_dir(DATASET_ROOT, "train")
LEGACY_TEST_DIR  = find_named_dir(DATASET_ROOT, "testing")  or find_named_dir(DATASET_ROOT, "test")

MIN_TEST_PER_CLASS = 20
def flat_image_count(d):
    if d is None or not os.path.isdir(d):
        return 0
    return len([f for f in os.listdir(d) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])

test_dir_has_subdirs = has_class_subdirs(LEGACY_TEST_DIR, CLASS_NAMES)
if test_dir_has_subdirs:
    test_img_count = sum(len([f for f in os.listdir(os.path.join(LEGACY_TEST_DIR, c))
                               if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])
                          for c in CLASS_NAMES)
else:
    test_img_count = flat_image_count(LEGACY_TEST_DIR)

USE_PROVIDED_TEST = test_dir_has_subdirs and (test_img_count / max(NUM_CLASSES, 1) >= MIN_TEST_PER_CLASS)
print(f"Provided test dir: {LEGACY_TEST_DIR} | has_subdirs={test_dir_has_subdirs} | "
      f"images={test_img_count} | usable={USE_PROVIDED_TEST}")

TRAIN_DIR = (LEGACY_TRAIN_DIR if (USE_PROVIDED_TEST and has_class_subdirs(LEGACY_TRAIN_DIR, CLASS_NAMES))
             else MAIN_DIR)
TEST_DIR = LEGACY_TEST_DIR if USE_PROVIDED_TEST else None
print("Final TRAIN_DIR:", TRAIN_DIR)
print("Final TEST_DIR :", TEST_DIR, "(None => self-splitting from MAIN_DIR)")


## 3. Build the labeled image index

In [ ]:
def list_images(d):
    rows = []
    for cls in CLASS_NAMES:
        p = os.path.join(d, cls)
        if not os.path.isdir(p):
            continue
        for f in os.listdir(p):
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                rows.append({"path": os.path.join(p, f), "label": cls})
    return pd.DataFrame(rows)

if USE_PROVIDED_TEST:
    train_df = list_images(TRAIN_DIR)
    test_df  = list_images(TEST_DIR)
    print("Train images:", len(train_df), " Test images (provided split):", len(test_df))
else:
    full_df = list_images(MAIN_DIR)
    train_df = full_df
    test_df = None
    print("Full labeled pool:", len(full_df), "images (will self-split 70/15/15)")

print(train_df["label"].value_counts())


## 4. Train / validation / test split

Split first, then audit for leakage against the *final* test set (auditing before the
split is meaningless if the split hasn't happened yet, which was the exact bug in an
earlier version of this pipeline).

In [ ]:
if USE_PROVIDED_TEST:
    train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df["label"], random_state=SEED)
else:
    train_df, temp_df = train_test_split(train_df, test_size=0.30, stratify=train_df["label"], random_state=SEED)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{name} label counts:\n{df['label'].value_counts()}\n")

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
train_df["label"].value_counts().plot(kind="bar", ax=ax[0], title="Train class distribution")
val_df["label"].value_counts().plot(kind="bar", ax=ax[1], title="Val class distribution")
test_df["label"].value_counts().plot(kind="bar", ax=ax[2], title="Test class distribution")
plt.tight_layout(); plt.savefig("figures/class_distribution.png", dpi=200); plt.show()


In [ ]:
# ---------------- Leakage audit: test set vs. everything used to fit the model ----------------
import imagehash

def hash_series(df):
    hashes = {}
    for _, row in df.iterrows():
        try:
            h = imagehash.phash(Image.open(row["path"]).convert("L"), hash_size=8)
            hashes[row["path"]] = h
        except Exception as e:
            print("skip", row["path"], e)
    return hashes

fit_pool_df = pd.concat([train_df, val_df], ignore_index=True)
fit_hashes = hash_series(fit_pool_df)
test_hashes = hash_series(test_df)

HAMMING_THRESH = 5
leak_pairs = []
for tp, th in test_hashes.items():
    for fp, fh in fit_hashes.items():
        if th - fh <= HAMMING_THRESH:
            leak_pairs.append((tp, fp, th - fh))

print(f"Potential test/(train+val) near-duplicates: {len(leak_pairs)} (threshold <= {HAMMING_THRESH})")
leak_df = pd.DataFrame(leak_pairs, columns=["test_path", "fit_path", "hamming_dist"])
leak_df.to_csv("leakage_audit.csv", index=False)

if len(leak_df) > 0:
    leaked_test_paths = set(leak_df["test_path"].unique())
    print(f"Removing {len(leaked_test_paths)} leaked images from the test set.")
    test_df = test_df[~test_df["path"].isin(leaked_test_paths)].reset_index(drop=True)
print("Final leak-audited test set size:", len(test_df))


## 5. Preprocessing: contour-based cropping + transforms

Every model in this notebook (both teachers and both from-scratch students) uses the same
contour-crop preprocessing: convert to grayscale, blur, threshold, find the largest
external contour, and crop to its bounding box before resizing. This removes background
and scanner-artifact regions from the MRI slice. Applying it uniformly to every model
keeps the head-to-head comparison fair; any accuracy contribution from cropping itself can
be read off by comparing this notebook's baseline GAMS-Net (with cropping) against the
prior notebook's GAMS-Net (without cropping, seed-averaged 89.39% ± 2.00%).

In [ ]:
def contour_crop(pil_img, pad=4):
    """Crop a PIL RGB image to the bounding box of its largest contour. Falls back to the
    original image on any failure (blank image, no contour found, etc.) so a single bad
    image can never crash a training epoch."""
    try:
        img = np.array(pil_img.convert("RGB"))
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        _, thresh = cv2.threshold(blurred, 15, 255, cv2.THRESH_BINARY)
        thresh = cv2.erode(thresh, None, iterations=2)
        thresh = cv2.dilate(thresh, None, iterations=2)
        contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return pil_img
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        if w < 10 or h < 10:
            return pil_img
        H, W = gray.shape
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(W, x + w + pad), min(H, y + h + pad)
        cropped = img[y0:y1, x0:x1]
        if cropped.size == 0:
            return pil_img
        return Image.fromarray(cropped)
    except Exception:
        return pil_img

class ContourCrop:
    def __call__(self, img):
        return contour_crop(img)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    ContourCrop(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tfms = transforms.Compose([
    ContourCrop(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class BrainMRIDataset(Dataset):
    def __init__(self, df, class_names, transform):
        self.df = df.reset_index(drop=True)
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        img = self.transform(img)
        label = self.class_to_idx[row["label"]]
        return img, label

def make_loaders(train_df, val_df, test_df, batch_size=BATCH_SIZE):
    train_ds = BrainMRIDataset(train_df, CLASS_NAMES, train_tfms)
    val_ds   = BrainMRIDataset(val_df,   CLASS_NAMES, eval_tfms)
    test_ds  = BrainMRIDataset(test_df,  CLASS_NAMES, eval_tfms)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader, test_ds

train_loader, val_loader, test_loader, test_ds = make_loaders(train_df, val_df, test_df)
print("Loaders ready:", len(train_df), "train /", len(val_df), "val /", len(test_df), "test")


## 6. Model definitions

`FlexGAMSNet` is a single, parameterized class covering both the original GAMS-Net
(`attention='eca', double_msdf=False`) and the proposed KD-GAMS-Net
(`attention='spatial', double_msdf=True`), plus every ablation variant in between —
verified in a sandbox to produce correct shapes, parameter counts (~2.96M for the original
configuration, ~3.16M for the proposed one), and working gradients before being placed in
this notebook.

In [ ]:
class ECA(nn.Module):
    """Efficient Channel Attention (Wang et al., 2020)."""
    def __init__(self, channels, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        return x * self.sigmoid(y)


class SpatialAttention(nn.Module):
    """CBAM-style spatial attention (Woo et al., 2018): gates on a 2D map derived from
    channel-wise average- and max-pooling, targeting *where* to look rather than *which
    channel* to trust."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        y = torch.cat([avg_out, max_out], dim=1)
        y = self.conv(y)
        return x * self.sigmoid(y)


def make_attention(kind, channels):
    if kind == "eca":
        return ECA(channels)
    elif kind == "spatial":
        return SpatialAttention()
    elif kind == "none":
        return nn.Identity()
    raise ValueError(f"unknown attention kind: {kind}")


class GhostModule(nn.Module):
    """Ghost module (Han et al., 2020): cheap linear ops generate 'ghost' feature maps."""
    def __init__(self, in_ch, out_ch, kernel_size=1, ratio=2, dw_size=3, stride=1, relu=True):
        super().__init__()
        init_ch = out_ch // ratio
        new_ch = out_ch - init_ch
        self.primary = nn.Sequential(
            nn.Conv2d(in_ch, init_ch, kernel_size, stride, kernel_size // 2, bias=False),
            nn.BatchNorm2d(init_ch), nn.ReLU(inplace=True) if relu else nn.Identity())
        self.cheap = nn.Sequential(
            nn.Conv2d(init_ch, new_ch, dw_size, 1, dw_size // 2, groups=init_ch, bias=False),
            nn.BatchNorm2d(new_ch), nn.ReLU(inplace=True) if relu else nn.Identity())
    def forward(self, x):
        y1 = self.primary(x)
        y2 = self.cheap(y1)
        return torch.cat([y1, y2], dim=1)


class GhostAttentionBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, attention="eca"):
        super().__init__()
        self.ghost1 = GhostModule(in_ch, out_ch, relu=True)
        self.dw = None
        if stride == 2:
            self.dw = nn.Sequential(
                nn.Conv2d(out_ch, out_ch, 3, stride, 1, groups=out_ch, bias=False),
                nn.BatchNorm2d(out_ch))
        self.ghost2 = GhostModule(out_ch, out_ch, relu=False)
        self.attn = make_attention(attention, out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False) if stride == 2 else nn.Identity(),
                nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=False), nn.BatchNorm2d(out_ch))
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.ghost1(x)
        if self.dw is not None:
            out = self.dw(out)
        out = self.ghost2(out)
        out = self.attn(out)
        return self.relu(out + residual)


class MSDFBlock(nn.Module):
    """Multi-Scale Dilated Fusion: parallel dilated convs (rates 1,2,3) fused by 1x1 conv."""
    def __init__(self, channels, dilations=(1, 2, 3), attention="eca"):
        super().__init__()
        branch_ch = channels // len(dilations)
        rem = channels - branch_ch * len(dilations)
        self.branches = nn.ModuleList()
        for i, d in enumerate(dilations):
            c = branch_ch + (rem if i == 0 else 0)
            self.branches.append(nn.Sequential(
                nn.Conv2d(channels, c, 3, 1, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(c), nn.ReLU(inplace=True)))
        self.project = nn.Sequential(nn.Conv2d(channels, channels, 1, 1, 0, bias=False), nn.BatchNorm2d(channels))
        self.attn = make_attention(attention, channels)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        out = torch.cat([b(x) for b in self.branches], dim=1)
        out = self.project(out)
        out = self.attn(out)
        return self.relu(out + x)


class FlexGAMSNet(nn.Module):
    """Unified GAMS-Net family. attention='eca', double_msdf=False reproduces the original
    GAMS-Net exactly. attention='spatial', double_msdf=True is the proposed KD-GAMS-Net
    architecture (trained with the KD+focal recipe defined in Section 7, not by this class)."""
    def __init__(self, num_classes=4, width=(72, 144, 288, 480, 672), dropout=0.25,
                 attention="eca", double_msdf=False):
        super().__init__()
        c0, c1, c2, c3, c4 = width
        self.stem = nn.Sequential(nn.Conv2d(3, c0, 3, 2, 1, bias=False), nn.BatchNorm2d(c0), nn.ReLU(inplace=True))
        self.stage1 = GhostAttentionBlock(c0, c1, stride=2, attention=attention)
        self.msdf1 = MSDFBlock(c1, attention=attention) if double_msdf else nn.Identity()
        self.stage2 = GhostAttentionBlock(c1, c2, stride=2, attention=attention)
        self.msdf2 = MSDFBlock(c2, attention=attention)
        self.stage3 = GhostAttentionBlock(c2, c3, stride=2, attention=attention)
        self.stage4 = GhostAttentionBlock(c3, c4, stride=2, attention=attention)
        self.head_conv = nn.Sequential(nn.Conv2d(c4, c4 * 2, 1, 1, 0, bias=False),
                                        nn.BatchNorm2d(c4 * 2), nn.ReLU(inplace=True))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(c4 * 2, num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.msdf1(x)
        x = self.stage2(x); x = self.msdf2(x)
        x = self.stage3(x); x = self.stage4(x)
        x = self.head_conv(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def build_teacher(name, num_classes):
    if name == "MobileNetV2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.last_channel, num_classes)
        return m
    if name == "EfficientNetB0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m
    raise ValueError(name)


# quick structural sanity check (also verified standalone before this notebook was written)
for tag, cfg in [("Original GAMS-Net", dict(attention='eca', double_msdf=False)),
                  ("KD-GAMS-Net (proposed)", dict(attention='spatial', double_msdf=True))]:
    m = FlexGAMSNet(num_classes=NUM_CLASSES, **cfg)
    n = count_params(m)
    print(f"{tag}: {n:,} trainable parameters ({n/1e6:.3f} M)")


## 7. Losses: Focal loss and Knowledge Distillation

`FocalLoss` upweights hard/rare examples and the notumor class specifically (its class
weight is set below in proportion to the training-set class imbalance). `KDLoss` combines
that hard-label focal loss with a temperature-scaled KL-divergence term against the frozen
teacher ensemble's softened output distribution.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
    def forward(self, logits, targets):
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()
        ce = F.nll_loss(logp, targets, weight=self.weight, reduction="none")
        pt = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()


class KDLoss(nn.Module):
    """alpha * hard_loss(student, labels) + (1 - alpha) * T^2 * KL(student_soft || teacher_soft)"""
    def __init__(self, hard_loss_fn, alpha=KD_ALPHA, T=KD_TEMPERATURE):
        super().__init__()
        self.hard_loss_fn = hard_loss_fn
        self.alpha = alpha
        self.T = T
    def forward(self, student_logits, teacher_logits, targets):
        hard = self.hard_loss_fn(student_logits, targets)
        student_log_soft = F.log_softmax(student_logits / self.T, dim=1)
        teacher_soft = F.softmax(teacher_logits / self.T, dim=1)
        soft = F.kl_div(student_log_soft, teacher_soft, reduction="batchmean") * (self.T ** 2)
        return self.alpha * hard + (1 - self.alpha) * soft


# class weights from the *training* set's class imbalance (inverse frequency, normalized)
class_counts = train_df["label"].value_counts().reindex(CLASS_NAMES)
inv_freq = (1.0 / class_counts)
class_weights_t = torch.tensor((inv_freq / inv_freq.mean()).values, dtype=torch.float32).to(DEVICE)
print("Class weights (by class order", CLASS_NAMES, "):", class_weights_t.tolist())


## 8. Training / evaluation / prediction-caching utilities

`evaluate_and_cache` is the fix for the previous notebook's confusion-matrix and
McNemar's-test gap: it is called immediately after every single model finishes training,
saves that model's true/predicted/probability arrays to `predictions/<tag>.npz`, and
immediately plots and saves that model's confusion matrix — so even if the Kaggle session
were interrupted later in the notebook, every model trained up to that point already has
its confusion matrix and cached predictions safely on disk.

In [ ]:
def train_model(model, train_loader, val_loader, loss_fn, epochs=EPOCHS, lr=1e-3,
                 patience=PATIENCE, tag="model", teacher=None):
    """teacher: None for plain hard-label training, or a callable(imgs)->averaged teacher
    logits (already in eval mode, no_grad) for KD training."""
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss, running_correct, n = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                out = model(imgs)
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = teacher(imgs)
                    loss = loss_fn(out, teacher_logits, labels)
                else:
                    loss = loss_fn(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)
            running_correct += (out.argmax(1) == labels).sum().item()
            n += imgs.size(0)
        train_loss, train_acc = running_loss / n, running_correct / n

        model.eval()
        v_loss, v_correct, vn = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                if teacher is not None:
                    teacher_logits = teacher(imgs)
                    loss = loss_fn(out, teacher_logits, labels)
                else:
                    loss = loss_fn(out, labels)
                v_loss += loss.item() * imgs.size(0)
                v_correct += (out.argmax(1) == labels).sum().item()
                vn += imgs.size(0)
        val_loss, val_acc = v_loss / vn, v_correct / vn
        scheduler.step(val_loss)
        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)
        print(f"[{tag}] epoch {epoch+1:02d}/{epochs}  train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
              f"train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{tag}] early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    elapsed = time.time() - t0
    print(f"[{tag}] training complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")
    return model, history, elapsed


@torch.no_grad()
def evaluate_and_cache(model, loader, tag, save=True):
    """Evaluate, cache y_true/y_pred/y_prob to predictions/<tag>.npz immediately, plot and
    save that model's confusion matrix immediately. Returns the metrics dict."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        out = model(imgs)
        probs = F.softmax(out, dim=1).cpu().numpy()
        all_preds.extend(probs.argmax(1).tolist())
        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.tolist())
    y_pred, y_true, y_prob = np.array(all_preds), np.array(all_labels), np.array(all_probs)

    acc = accuracy_score(y_true, y_pred)
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)

    if save:
        np.savez(f"predictions/{tag}.npz", y_true=y_true, y_pred=y_pred, y_prob=y_prob)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES,
                    yticklabels=CLASS_NAMES, ax=ax, cbar=False)
        ax.set_title(tag); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        plt.tight_layout(); plt.savefig(f"figures/confusion_{tag}.png", dpi=200); plt.show(); plt.close(fig)
        print(f"[{tag}] cached predictions -> predictions/{tag}.npz, confusion matrix -> figures/confusion_{tag}.png")

    print(f"\n=== {tag} ===\n{report}")
    return {"accuracy": acc, "precision_macro": prec_m, "recall_macro": rec_m, "f1_macro": f1_m,
            "precision_weighted": prec_w, "recall_weighted": rec_w, "f1_weighted": f1_w,
            "auc_macro_ovr": auc, "confusion_matrix": cm, "y_true": y_true, "y_pred": y_pred, "y_prob": y_prob}


@torch.no_grad()
def measure_latency(model, input_size=(1, 3, IMG_SIZE, IMG_SIZE), n_iters=50, device=DEVICE):
    model = model.to(device).eval()
    x = torch.randn(*input_size).to(device)
    for _ in range(10):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_iters):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / n_iters * 1000


def real_mcnemar(tag_a, tag_b):
    """Genuine paired McNemar's test from two cached prediction files on the SAME test set
    (fixes the previous notebook's inability to compute this: per-sample predictions are now
    always cached by evaluate_and_cache, so this never has to fall back to a seed-level
    approximation)."""
    da = np.load(f"predictions/{tag_a}.npz")
    db = np.load(f"predictions/{tag_b}.npz")
    assert np.array_equal(da["y_true"], db["y_true"]), f"{tag_a} and {tag_b} were not evaluated on the same test set!"
    y_true = da["y_true"]
    correct_a = (da["y_pred"] == y_true)
    correct_b = (db["y_pred"] == y_true)
    both_correct = int(np.sum(correct_a & correct_b))
    a_only = int(np.sum(correct_a & ~correct_b))
    b_only = int(np.sum(~correct_a & correct_b))
    both_wrong = int(np.sum(~correct_a & ~correct_b))
    table = [[both_correct, a_only], [b_only, both_wrong]]
    result = mcnemar(table, exact=(a_only + b_only) < 25, correction=True)
    return {"a": tag_a, "b": tag_b, "acc_a": correct_a.mean(), "acc_b": correct_b.mean(),
            "table": table, "statistic": result.statistic, "p_value": result.pvalue}


## 9. Train the teacher ensemble (seed 42)

Both teachers are fine-tuned once at seed 42 and then frozen; they are reused as a fixed
KD source across every student run in this notebook (including the multi-seed sweep in
Section 12), since the object of study is the *student's* sensitivity to training recipe
and initialization — not the teachers'.

In [ ]:
set_seed(SEED)
mobilenet = build_teacher("MobileNetV2", NUM_CLASSES)
mobilenet, hist_mob, t_mob = train_model(mobilenet, train_loader, val_loader, nn.CrossEntropyLoss(),
                                          lr=1e-4, tag="MobileNetV2-seed42")
res_mobilenet = evaluate_and_cache(mobilenet, test_loader, "MobileNetV2-seed42")
torch.save(mobilenet.state_dict(), "checkpoints/MobileNetV2-seed42.pt")


In [ ]:
set_seed(SEED)
effnet = build_teacher("EfficientNetB0", NUM_CLASSES)
effnet, hist_eff, t_eff = train_model(effnet, train_loader, val_loader, nn.CrossEntropyLoss(),
                                       lr=1e-4, tag="EfficientNetB0-seed42")
res_effnet = evaluate_and_cache(effnet, test_loader, "EfficientNetB0-seed42")
torch.save(effnet.state_dict(), "checkpoints/EfficientNetB0-seed42.pt")


In [ ]:
class TeacherEnsemble:
    """Frozen, eval-mode wrapper averaging two teachers' logits for KD."""
    def __init__(self, models_list):
        self.models = [m.eval() for m in models_list]
        for m in self.models:
            for p in m.parameters():
                p.requires_grad_(False)
    def __call__(self, imgs):
        logits = [m(imgs) for m in self.models]
        return torch.stack(logits, dim=0).mean(dim=0)

teacher_ensemble = TeacherEnsemble([mobilenet, effnet])
print("Teacher ensemble ready (MobileNetV2 + EfficientNet-B0, frozen).")


## 10. Train the headline students (seed 42): GAMS-Net baseline vs. KD-GAMS-Net

The baseline reproduces the original GAMS-Net recipe exactly (ECA, single MSDF, plain
cross-entropy, no distillation) so that any difference against KD-GAMS-Net is attributable
to the proposed changes and not to some other confound.

In [ ]:
set_seed(SEED)
gams_baseline = FlexGAMSNet(num_classes=NUM_CLASSES, attention="eca", double_msdf=False)
gams_baseline, hist_base, t_base = train_model(
    gams_baseline, train_loader, val_loader, nn.CrossEntropyLoss(),
    lr=1e-3, tag="GAMS-Net-baseline-seed42")
res_gams_baseline = evaluate_and_cache(gams_baseline, test_loader, "GAMS-Net-baseline-seed42")
torch.save(gams_baseline.state_dict(), "checkpoints/GAMS-Net-baseline-seed42.pt")


In [ ]:
set_seed(SEED)
kd_gams = FlexGAMSNet(num_classes=NUM_CLASSES, attention="spatial", double_msdf=True)
kd_loss_fn = KDLoss(hard_loss_fn=FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights_t),
                     alpha=KD_ALPHA, T=KD_TEMPERATURE)
kd_gams, hist_kd, t_kd = train_model(
    kd_gams, train_loader, val_loader, kd_loss_fn,
    lr=1e-3, tag="KD-GAMS-Net-seed42", teacher=teacher_ensemble)
res_kd_gams = evaluate_and_cache(kd_gams, test_loader, "KD-GAMS-Net-seed42")
torch.save(kd_gams.state_dict(), "checkpoints/KD-GAMS-Net-seed42.pt")

print(f"\nSeed-42 headline comparison: GAMS-Net baseline = {res_gams_baseline['accuracy']:.4f}  "
      f"|  KD-GAMS-Net = {res_kd_gams['accuracy']:.4f}  "
      f"|  MobileNetV2 = {res_mobilenet['accuracy']:.4f}  |  EfficientNet-B0 = {res_effnet['accuracy']:.4f}")


## 11. Ablation (single seed 42): isolating KD, double-MSDF, and spatial attention

Three variants, each changing exactly one element of the "full" KD-GAMS-Net recipe:

- **w/o KD**: same architecture (spatial attention, double MSDF), trained with focal loss
  only — isolates distillation's contribution.
- **w/o double-MSDF**: KD + focal + spatial attention, but MSDF only in its original single
  position — isolates whether doubling MSDF actually helped.
- **w/o spatial attention**: KD + focal + double MSDF, but ECA instead of spatial attention
  — isolates the channel-vs-spatial attention swap.

Kept at a single seed (like the original notebook's ablation) to stay within the runtime
budget; the accompanying paper should flag this the same way the prior paper did.

In [ ]:
set_seed(SEED)
ablation_wo_kd = FlexGAMSNet(num_classes=NUM_CLASSES, attention="spatial", double_msdf=True)
focal_only = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights_t)
ablation_wo_kd, _, t_wokd = train_model(ablation_wo_kd, train_loader, val_loader, focal_only,
                                         lr=1e-3, tag="ablation-wo-KD-seed42")
res_wo_kd = evaluate_and_cache(ablation_wo_kd, test_loader, "ablation-wo-KD-seed42")


In [ ]:
set_seed(SEED)
ablation_wo_dmsdf = FlexGAMSNet(num_classes=NUM_CLASSES, attention="spatial", double_msdf=False)
kd_loss_fn2 = KDLoss(hard_loss_fn=FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights_t), alpha=KD_ALPHA, T=KD_TEMPERATURE)
ablation_wo_dmsdf, _, t_wodmsdf = train_model(ablation_wo_dmsdf, train_loader, val_loader, kd_loss_fn2,
                                               lr=1e-3, tag="ablation-wo-doubleMSDF-seed42", teacher=teacher_ensemble)
res_wo_dmsdf = evaluate_and_cache(ablation_wo_dmsdf, test_loader, "ablation-wo-doubleMSDF-seed42")


In [ ]:
set_seed(SEED)
ablation_wo_spatial = FlexGAMSNet(num_classes=NUM_CLASSES, attention="eca", double_msdf=True)
kd_loss_fn3 = KDLoss(hard_loss_fn=FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights_t), alpha=KD_ALPHA, T=KD_TEMPERATURE)
ablation_wo_spatial, _, t_wospatial = train_model(ablation_wo_spatial, train_loader, val_loader, kd_loss_fn3,
                                                   lr=1e-3, tag="ablation-wo-spatial-seed42", teacher=teacher_ensemble)
res_wo_spatial = evaluate_and_cache(ablation_wo_spatial, test_loader, "ablation-wo-spatial-seed42")


In [ ]:
ablation_rows = [
    {"Variant": "Full KD-GAMS-Net", "Params(M)": round(count_params(kd_gams)/1e6, 3), "Accuracy": res_kd_gams["accuracy"], "F1": res_kd_gams["f1_macro"]},
    {"Variant": "w/o KD", "Params(M)": round(count_params(ablation_wo_kd)/1e6, 3), "Accuracy": res_wo_kd["accuracy"], "F1": res_wo_kd["f1_macro"]},
    {"Variant": "w/o double-MSDF", "Params(M)": round(count_params(ablation_wo_dmsdf)/1e6, 3), "Accuracy": res_wo_dmsdf["accuracy"], "F1": res_wo_dmsdf["f1_macro"]},
    {"Variant": "w/o spatial attn (ECA)", "Params(M)": round(count_params(ablation_wo_spatial)/1e6, 3), "Accuracy": res_wo_spatial["accuracy"], "F1": res_wo_spatial["f1_macro"]},
]
ablation_df = pd.DataFrame(ablation_rows)
ablation_df["Delta_Accuracy_vs_Full"] = (ablation_df["Accuracy"] - ablation_df.loc[0, "Accuracy"]).round(4)
ablation_df.to_csv("ablation_results.csv", index=False)
ablation_df


## 12. Multi-seed robustness (seeds 123, 2024) for the 4 headline models

Seed 42 is already done above. This section re-trains the 4 headline models (GAMS-Net
baseline, KD-GAMS-Net, MobileNetV2, EfficientNet-B0) at two more seeds. The frozen
seed-42 teacher ensemble is reused as the KD source at every seed (see Section 9's
rationale). Set `SKIP_MULTISEED = True` in Section 1 to skip this and save ~4–5 hours.

In [ ]:
seed_results = {
    "GAMS-Net-baseline": [res_gams_baseline["accuracy"]],
    "KD-GAMS-Net": [res_kd_gams["accuracy"]],
    "MobileNetV2": [res_mobilenet["accuracy"]],
    "EfficientNetB0": [res_effnet["accuracy"]],
}
seed_results_f1 = {
    "GAMS-Net-baseline": [res_gams_baseline["f1_macro"]],
    "KD-GAMS-Net": [res_kd_gams["f1_macro"]],
    "MobileNetV2": [res_mobilenet["f1_macro"]],
    "EfficientNetB0": [res_effnet["f1_macro"]],
}

if not SKIP_MULTISEED:
    for extra_seed in SEEDS_EXTRA:
        print(f"\n{'='*30} SEED {extra_seed} {'='*30}")

        set_seed(extra_seed)
        m_base = FlexGAMSNet(num_classes=NUM_CLASSES, attention="eca", double_msdf=False)
        m_base, _, _ = train_model(m_base, train_loader, val_loader, nn.CrossEntropyLoss(),
                                    lr=1e-3, tag=f"GAMS-Net-baseline-seed{extra_seed}")
        r_base = evaluate_and_cache(m_base, test_loader, f"GAMS-Net-baseline-seed{extra_seed}", save=False)
        seed_results["GAMS-Net-baseline"].append(r_base["accuracy"])
        seed_results_f1["GAMS-Net-baseline"].append(r_base["f1_macro"])

        set_seed(extra_seed)
        m_kd = FlexGAMSNet(num_classes=NUM_CLASSES, attention="spatial", double_msdf=True)
        kd_loss_fn_s = KDLoss(hard_loss_fn=FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights_t), alpha=KD_ALPHA, T=KD_TEMPERATURE)
        m_kd, _, _ = train_model(m_kd, train_loader, val_loader, kd_loss_fn_s,
                                  lr=1e-3, tag=f"KD-GAMS-Net-seed{extra_seed}", teacher=teacher_ensemble)
        r_kd = evaluate_and_cache(m_kd, test_loader, f"KD-GAMS-Net-seed{extra_seed}", save=False)
        seed_results["KD-GAMS-Net"].append(r_kd["accuracy"])
        seed_results_f1["KD-GAMS-Net"].append(r_kd["f1_macro"])

        set_seed(extra_seed)
        m_mob = build_teacher("MobileNetV2", NUM_CLASSES)
        m_mob, _, _ = train_model(m_mob, train_loader, val_loader, nn.CrossEntropyLoss(),
                                   lr=1e-4, tag=f"MobileNetV2-seed{extra_seed}")
        r_mob = evaluate_and_cache(m_mob, test_loader, f"MobileNetV2-seed{extra_seed}", save=False)
        seed_results["MobileNetV2"].append(r_mob["accuracy"])
        seed_results_f1["MobileNetV2"].append(r_mob["f1_macro"])

        set_seed(extra_seed)
        m_eff = build_teacher("EfficientNetB0", NUM_CLASSES)
        m_eff, _, _ = train_model(m_eff, train_loader, val_loader, nn.CrossEntropyLoss(),
                                   lr=1e-4, tag=f"EfficientNetB0-seed{extra_seed}")
        r_eff = evaluate_and_cache(m_eff, test_loader, f"EfficientNetB0-seed{extra_seed}", save=False)
        seed_results["EfficientNetB0"].append(r_eff["accuracy"])
        seed_results_f1["EfficientNetB0"].append(r_eff["f1_macro"])
else:
    print("SKIP_MULTISEED=True: only the seed-42 results above are available.")


In [ ]:
robust_rows = []
for name in seed_results:
    accs = np.array(seed_results[name]); f1s = np.array(seed_results_f1[name])
    robust_rows.append({
        "Model": name, "N_seeds": len(accs),
        "Accuracy_mean": accs.mean(), "Accuracy_std": accs.std(),
        "F1_mean": f1s.mean(), "F1_std": f1s.std(),
    })
robustness_df = pd.DataFrame(robust_rows)
robustness_df.to_csv("seed_robustness.csv", index=False)
robustness_df


In [ ]:
if not SKIP_MULTISEED and len(seed_results["KD-GAMS-Net"]) >= 3:
    print("Welch's t-test: KD-GAMS-Net vs. each other headline model (n=3 seeds per group; low-power, indicative only)\n")
    kd_accs = np.array(seed_results["KD-GAMS-Net"])
    for name in ["GAMS-Net-baseline", "MobileNetV2", "EfficientNetB0"]:
        other = np.array(seed_results[name])
        t, p = scipy_stats.ttest_ind(kd_accs, other, equal_var=False)
        direction = "higher" if kd_accs.mean() > other.mean() else "lower"
        print(f"KD-GAMS-Net ({kd_accs.mean():.4f}) vs {name} ({other.mean():.4f}): "
              f"t={t:.3f}, p={p:.4f}  [KD-GAMS-Net is {direction}]")
else:
    print("Multi-seed comparison skipped or insufficient seeds; see Section 13 for the single-seed McNemar's tests instead.")


## 13. McNemar's test (real, per-sample, seed 42)

This is the fix for the previous notebook's missing McNemar's results: every model above
was evaluated with `evaluate_and_cache`, which always saves true/predicted labels to disk.
The test below loads those cached arrays directly — no approximation, no seed-level
workaround, a genuine paired test on identical test-set samples.

In [ ]:
mcnemar_pairs = [
    ("KD-GAMS-Net-seed42", "GAMS-Net-baseline-seed42"),
    ("KD-GAMS-Net-seed42", "MobileNetV2-seed42"),
    ("KD-GAMS-Net-seed42", "EfficientNetB0-seed42"),
    ("KD-GAMS-Net-seed42", "ablation-wo-KD-seed42"),
    ("KD-GAMS-Net-seed42", "ablation-wo-doubleMSDF-seed42"),
    ("KD-GAMS-Net-seed42", "ablation-wo-spatial-seed42"),
]

mcnemar_rows = []
for a, b in mcnemar_pairs:
    r = real_mcnemar(a, b)
    mcnemar_rows.append({
        "Comparison": f"{a} vs {b}",
        "Acc_A": round(r["acc_a"], 4), "Acc_B": round(r["acc_b"], 4),
        "Statistic": round(r["statistic"], 3), "p_value": round(r["p_value"], 5),
        "Significant_at_0.05": "Yes" if r["p_value"] < 0.05 else "No",
    })
    print(f"{a} vs {b}: acc_A={r['acc_a']:.4f} acc_B={r['acc_b']:.4f} "
          f"statistic={r['statistic']:.3f} p={r['p_value']:.5f} table={r['table']}")

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv("mcnemar_results.csv", index=False)
mcnemar_df


## 14. Combined confusion matrix grid (headline models, seed 42)

In [ ]:
headline_tags = ["KD-GAMS-Net-seed42", "GAMS-Net-baseline-seed42", "MobileNetV2-seed42", "EfficientNetB0-seed42"]
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, tag in zip(axes.flat, headline_tags):
    d = np.load(f"predictions/{tag}.npz")
    cm = confusion_matrix(d["y_true"], d["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, cbar=False)
    acc = (d["y_pred"] == d["y_true"]).mean()
    ax.set_title(f"{tag}  (acc={acc:.4f})"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig("figures/confusion_grid_headline.png", dpi=200); plt.show()


## 15. Efficiency comparison (params, FLOPs, latency)

In [ ]:
from thop import profile as thop_profile

models_for_efficiency = {
    "KD-GAMS-Net": kd_gams, "GAMS-Net-baseline": gams_baseline,
    "MobileNetV2": mobilenet, "EfficientNetB0": effnet,
}
eff_rows = []
for name, m in models_for_efficiency.items():
    m = m.to(DEVICE)
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    macs, _ = thop_profile(copy.deepcopy(m), inputs=(dummy,), verbose=False)
    lat_gpu = measure_latency(m, device=DEVICE) if DEVICE.type == "cuda" else float("nan")
    lat_cpu = measure_latency(m, device=torch.device("cpu"))
    eff_rows.append({"Model": name, "Params(M)": round(count_params(m)/1e6, 3),
                      "FLOPs(GMac)": round(macs/1e9, 3),
                      "Latency_GPU_ms": round(lat_gpu, 3) if lat_gpu == lat_gpu else None,
                      "Latency_CPU_ms": round(lat_cpu, 3)})
efficiency_df = pd.DataFrame(eff_rows)
efficiency_df.to_csv("efficiency_results.csv", index=False)

# measure_latency's CPU-latency pass moves each model onto the CPU in-place (nn.Module.to()
# mutates and returns self); restore every model to DEVICE now so later cells (Grad-CAM in
# Section 16) don't hit a device-mismatch error.
for _m in models_for_efficiency.values():
    _m.to(DEVICE)
print("Models restored to", DEVICE, "after latency benchmarking.")

efficiency_df


## 16. Grad-CAM (KD-GAMS-Net)

Self-installing (survives a fresh kernel session even if Section 1's install cell wasn't
re-run), and passes an explicit `targets=` argument as required by current `grad-cam`
package versions.

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "grad-cam"], check=True)
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

target_layer = kd_gams.stage4
cam = GradCAM(model=kd_gams, target_layers=[target_layer])

def unnormalize(img_tensor):
    img = img_tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

n_show = 8
idxs = np.random.choice(len(test_ds), n_show, replace=False)
fig, axes = plt.subplots(2, n_show, figsize=(3*n_show, 6))
for i, idx in enumerate(idxs):
    img_tensor, label = test_ds[idx]
    input_tensor = img_tensor.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = kd_gams(input_tensor).argmax(1).item()
    targets = [ClassifierOutputTarget(pred)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
    rgb_img = unnormalize(img_tensor)
    vis = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
    axes[0, i].imshow(rgb_img); axes[0, i].axis("off"); axes[0, i].set_title(f"true: {CLASS_NAMES[label]}", fontsize=9)
    axes[1, i].imshow(vis); axes[1, i].axis("off")
    color = "green" if pred == label else "red"
    axes[1, i].set_title(f"pred: {CLASS_NAMES[pred]}", fontsize=9, color=color)
plt.tight_layout(); plt.savefig("figures/gradcam_kd_gams_net.png", dpi=200); plt.show()


## 17. Save summary artifacts

In [ ]:
summary = {
    "seed42_headline_accuracy": {
        "KD-GAMS-Net": res_kd_gams["accuracy"], "GAMS-Net-baseline": res_gams_baseline["accuracy"],
        "MobileNetV2": res_mobilenet["accuracy"], "EfficientNetB0": res_effnet["accuracy"],
    },
    "params_M": {k: round(count_params(v)/1e6, 3) for k, v in
                 [("KD-GAMS-Net", kd_gams), ("GAMS-Net-baseline", gams_baseline),
                  ("MobileNetV2", mobilenet), ("EfficientNetB0", effnet)]},
    "config": {"EPOCHS": EPOCHS, "PATIENCE": PATIENCE, "KD_ALPHA": KD_ALPHA,
               "KD_TEMPERATURE": KD_TEMPERATURE, "FOCAL_GAMMA": FOCAL_GAMMA,
               "SKIP_MULTISEED": SKIP_MULTISEED, "seeds": [SEED] + ([] if SKIP_MULTISEED else SEEDS_EXTRA)},
}
with open("run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved: ablation_results.csv, seed_robustness.csv, mcnemar_results.csv, efficiency_results.csv, "
      "leakage_audit.csv, run_summary.json")
print("Saved figures/: class_distribution.png, confusion_<tag>.png (one per model), "
      "confusion_grid_headline.png, gradcam_kd_gams_net.png")
print("Saved predictions/: <tag>.npz for every model (real y_true/y_pred/y_prob, reusable without retraining)")
print("Saved checkpoints/: seed-42 .pt weights for the 4 headline models")
print()
print("=== HEADLINE RESULT ===")
print(f"KD-GAMS-Net: {res_kd_gams['accuracy']:.4f}  vs  GAMS-Net baseline: {res_gams_baseline['accuracy']:.4f}"
      f"  (delta: {res_kd_gams['accuracy'] - res_gams_baseline['accuracy']:+.4f})")


## 18. Summary checklist before writing the paper

- [ ] Confirm `leakage_audit.csv` — report exact number of near-duplicate pairs found/removed
- [ ] Use `run_summary.json` + Section 10 output for the seed-42 headline comparison
- [ ] Use `seed_robustness.csv` (mean ± std) as the primary reported result, not the single seed-42 run
- [ ] Use `mcnemar_results.csv` for the **real**, paired, per-sample significance claims — this notebook
      fixes the previous version's gap by caching predictions immediately after every model trains
- [ ] Use `ablation_results.csv` to report which of KD / double-MSDF / spatial-attention actually helped —
      report all three honestly, including any that don't clear GAMS-Net-baseline's noise floor
- [ ] Use `efficiency_results.csv` for the params/FLOPs/latency table
- [ ] Include the confusion matrix grid (`figures/confusion_grid_headline.png`) and at least one
      Grad-CAM panel (`figures/gradcam_kd_gams_net.png`)
- [ ] If `SKIP_MULTISEED` was used, say so explicitly and treat all numbers as single-seed
- [ ] State exact package versions and hardware (GPU model) for reproducibility
